# JFCNN Training — Productive vs Waste

Trains the Joint Semantic + Structural CNN on the labeled page corpus.

**Pipeline**
1. Load labeled records from SQLite, drop `skip`
2. Build vocab + GloVe / structural embedding matrices
3. Stratified 80/20 train/val split
4. Train with Adam + CrossEntropyLoss
5. Evaluate: accuracy, precision, recall, F1, AUC
6. Save model weights + vocab artifacts to `checkpoints/`

> **Note — small dataset**: after dropping `skip`, the corpus currently has ~19 pages.
> All metrics on a 4-sample val set are high-variance. Treat them as sanity checks
> until the corpus grows.

In [20]:
import sys
sys.path.insert(0, '.')

import json
import os
import pickle
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

import config
from semantic_structure.db import load_records
from semantic_structure.extractor import build_corpus_vocab
from semantic_structure.dataset import PageDataset
from semantic_structure.model import JFCNN

print('Imports OK')

Imports OK


## Hyperparameters

In [ ]:
# ── Data ───────────────────────────────────────────────────────────────────
VAL_SPLIT    = 0.10      # fraction of data held out for validation
TEST_SPLIT   = 0.20      # fraction of data held out for final test
SEED         = 42

# ── Model ──────────────────────────────────────────────────────────────────
NUM_FILTERS  = 128       # conv filters per kernel size
KERNEL_SIZES = [3, 4, 5] # parallel conv towers
FC_HIDDEN    = 256
DROPOUT      = 0.5
# TODO: set FREEZE_WORD_EMB=False to fine-tune GloVe weights once the
#       corpus is large enough (>1 k labeled pages recommended).
FREEZE_WORD_EMB = True

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE   = 8
NUM_EPOCHS   = 100
LR           = 1e-3
WD           = 1e-4

# ── Paths ──────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = os.path.join(os.path.dirname(os.path.abspath('.')), 'models', 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f'Device:          {DEVICE}')
print(f'Checkpoint dir:  {CHECKPOINT_DIR}')

Device:          mps
Checkpoint dir:  /Users/umair/mlops/models/checkpoints


## 1. Load data + build vocab

In [22]:
from collections import Counter

all_records = load_records(config.DB_PATH)
print(f'Total labeled records: {len(all_records)}')
print(f'Label breakdown: {dict(Counter(r[2] for r in all_records))}')

# build_corpus_vocab uses ALL records (including skip) for vocabulary coverage,
# then PageDataset drops skip rows at construction time.
token2idx, word_matrix, struct_matrix = build_corpus_vocab(
    all_records, config.GLOVE_PATH, config.N, config.M
)

k = word_matrix.shape[1]
print(f'\nVocab size:  {len(token2idx)}')
print(f'k (GloVe):   {k}')
print(f'n (struct):  {config.N}')
print(f'Input dim:   {k + config.N}  (k+n per token)')

Total labeled records: 900
Label breakdown: {'waste': 429, 'productive': 470, 'skip': 1}
Parsing HTML corpus ...


100%|██████████| 900/900 [00:05<00:00, 163.13page/s]


Loading GloVe from /Users/umair/mlops/models/semantic_structure/GloVe/dolma_300_2024_1.2M.100_combined.txt ...


1200000 lines [00:10, 116872.87 lines/s]

GloVe dimension: 300
Vocab: 20146 tokens | GloVe coverage: 17395/20144 (86.4%)

Vocab size:  20146
k (GloVe):   300
n (struct):  16
Input dim:   316  (k+n per token)


## 2. Dataset + stratified train/val split

In [23]:
dataset = PageDataset(
    records   = all_records,
    token2idx = token2idx,
    tag2idx   = config.TAG_TO_IDX,
    label2idx = config.LABEL_TO_IDX,
    m         = config.M,
)

print(f'Dataset size after dropping skip: {len(dataset)}')
label_counts = Counter(dataset.records[i][2] for i in range(len(dataset)))
print(f'Label breakdown: {dict(label_counts)}')

# Stratified 70/10/20 train/val/test split
rng = random.Random(SEED)

by_class = {}
for i, (_, _, label_str) in enumerate(dataset.records):
    by_class.setdefault(label_str, []).append(i)

train_indices, val_indices, test_indices = [], [], []
for label_str, indices in by_class.items():
    rng.shuffle(indices)
    n_test = max(1, round(len(indices) * TEST_SPLIT))
    n_val  = max(1, round(len(indices) * VAL_SPLIT))
    test_indices.extend(indices[:n_test])
    val_indices.extend(indices[n_test:n_test + n_val])
    train_indices.extend(indices[n_test + n_val:])

rng.shuffle(train_indices)
rng.shuffle(val_indices)
rng.shuffle(test_indices)

train_set = Subset(dataset, train_indices)
val_set   = Subset(dataset, val_indices)
test_set  = Subset(dataset, test_indices)

print(f'\nTrain: {len(train_set)} samples (70%)')
print(f'Val:   {len(val_set)} samples (10%)')
print(f'Test:  {len(test_set)} samples (20%)')

Dataset size after dropping skip: 899
Label breakdown: {'waste': 429, 'productive': 470}

Train: 629 samples (70%)
Val:   90 samples (10%)
Test:  180 samples (20%)


In [24]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

Train batches: 79
Val batches:   12
Test batches:  23


## 3. Build model

In [25]:
torch.manual_seed(SEED)

model = JFCNN(
    word_matrix    = word_matrix,
    struct_matrix  = struct_matrix,
    num_filters    = NUM_FILTERS,
    kernel_sizes   = KERNEL_SIZES,
    fc_hidden      = FC_HIDDEN,
    dropout        = DROPOUT,
    num_classes    = 2,
    freeze_word_emb = FREEZE_WORD_EMB,
).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(model)
print(f'\nTotal params:     {total:,}')
print(f'Trainable params: {trainable:,}')

JFCNN(
  (word_emb): Embedding(20146, 300, padding_idx=0)
  (struct_emb): Embedding(13, 16, padding_idx=0)
  (convs): ModuleList(
    (0): Conv1d(316, 128, kernel_size=(3,), stride=(1,))
    (1): Conv1d(316, 128, kernel_size=(4,), stride=(1,))
    (2): Conv1d(316, 128, kernel_size=(5,), stride=(1,))
  )
  (relu): ReLU()
  (dropout): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=384, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=2, bias=True)
)

Total params:     6,628,842
Trainable params: 585,042


## 4. Optimizer + loss

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
criterion = nn.CrossEntropyLoss()  # applies log-softmax internally

## 5. Training loop

In [27]:
def evaluate(model, loader, criterion, device):
    """Run one pass over loader, return loss + all metrics."""
    model.eval()
    all_labels = []
    all_preds  = []
    all_probs  = []   # P(waste) for AUC
    total_loss = 0.0

    with torch.no_grad():
        for word_idx, tag_idx, labels in loader:
            word_idx = word_idx.to(device)
            tag_idx  = tag_idx.to(device)
            labels   = labels.to(device)

            logits = model(word_idx, tag_idx)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * len(labels)

            probs = torch.softmax(logits, dim=-1)[:, 1]  # P(waste)
            preds = logits.argmax(dim=-1)

            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())

    n = len(all_labels)
    avg_loss  = total_loss / n
    accuracy  = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall    = recall_score(all_labels, all_preds, zero_division=0)
    f1        = f1_score(all_labels, all_preds, zero_division=0)
    # AUC requires at least one sample from each class in the split
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    return dict(loss=avg_loss, accuracy=accuracy,
                precision=precision, recall=recall, f1=f1, auc=auc)


history = []

print(f'{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>8}  {'Acc':>6}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}  {'AUC':>6}')
print('-' * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for word_idx, tag_idx, labels in train_loader:
        word_idx = word_idx.to(DEVICE)
        tag_idx  = tag_idx.to(DEVICE)
        labels   = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(word_idx, tag_idx)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)

    train_loss /= len(train_set)

    # ── Validate ───────────────────────────────────────────────────────────
    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    history.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics})

    if epoch % 5 == 0 or epoch == 1:
        print(
            f'{epoch:>6}  {train_loss:>10.4f}  '
            f'{val_metrics["loss"]:>8.4f}  '
            f'{val_metrics["accuracy"]:>6.3f}  '
            f'{val_metrics["precision"]:>6.3f}  '
            f'{val_metrics["recall"]:>6.3f}  '
            f'{val_metrics["f1"]:>6.3f}  '
            f'{val_metrics["auc"]:>6.3f}'
        )

print('\nTraining complete.')

 Epoch  Train Loss  Val Loss     Acc    Prec     Rec      F1     AUC
---------------------------------------------------------------------------
     1      0.6044    0.3542   0.900   0.854   0.953   0.901   0.948
     5      0.2107    0.3063   0.867   0.804   0.953   0.872   0.951
    10      0.1739    0.3689   0.900   0.886   0.907   0.897   0.951
    15      0.1372    0.3327   0.911   0.872   0.953   0.911   0.954
    20      0.1312    0.3597   0.911   0.889   0.930   0.909   0.961
    25      0.1213    0.3487   0.900   0.870   0.930   0.899   0.957
    30      0.1223    0.3558   0.900   0.854   0.953   0.901   0.958
    35      0.1386    0.3794   0.878   0.833   0.930   0.879   0.952
    40      0.1311    0.4790   0.867   0.816   0.930   0.870   0.944
    45      0.1499    0.8143   0.856   0.778   0.977   0.866   0.926
    50      0.1287    0.7131   0.889   0.824   0.977   0.894   0.939
    55      0.1503    0.7594   0.856   0.778   0.977   0.866   0.921
    60      0.1275    0.585

## 6. Final evaluation

In [30]:
final = evaluate(model, test_loader, criterion, DEVICE)

print('Test metrics (final epoch)')
print('-' * 30)
for name, test in final.items():
    print(f'  {name:12s}: {test:.4f}')

Test metrics (final epoch)
------------------------------
  loss        : 0.9164
  accuracy    : 0.8944
  precision   : 0.8941
  recall      : 0.8837
  f1          : 0.8889
  auc         : 0.9474


## 7. Save model + vocab artifacts

In [29]:
test_metrics = evaluate(model, test_loader, criterion, DEVICE)

print('Test metrics (held-out set)')
print('-' * 30)
for name, val in test_metrics.items():
    print(f'  {name:12s}: {val:.4f}')

# Model weights
model_path = os.path.join(CHECKPOINT_DIR, 'jfcnn.pt')
torch.save(model.state_dict(), model_path)
print(f'\nModel weights saved → {model_path}')

# token2idx (needed to tokenise new pages at inference time)
vocab_path = os.path.join(CHECKPOINT_DIR, 'token2idx.json')
with open(vocab_path, 'w') as f:
    json.dump(token2idx, f)
print(f'token2idx saved    → {vocab_path}')

# Embedding matrices (word_matrix is large; struct_matrix is trained)
emb_path = os.path.join(CHECKPOINT_DIR, 'embeddings.npz')
np.savez(emb_path, word_matrix=word_matrix, struct_matrix=struct_matrix)
print(f'Embeddings saved   → {emb_path}')

# Training config for reproducibility
cfg_path = os.path.join(CHECKPOINT_DIR, 'train_config.json')
with open(cfg_path, 'w') as f:
    json.dump(dict(
        num_filters=NUM_FILTERS,
        kernel_sizes=KERNEL_SIZES,
        fc_hidden=FC_HIDDEN,
        dropout=DROPOUT,
        freeze_word_emb=FREEZE_WORD_EMB,
        num_classes=2,
        batch_size=BATCH_SIZE,
        num_epochs=NUM_EPOCHS,
        lr=LR,
        seed=SEED,
        val_split=VAL_SPLIT,
        test_split=TEST_SPLIT,
        final_val_metrics=final,
        final_test_metrics=test_metrics,
    ), f, indent=2)
print(f'Train config saved → {cfg_path}')

Test metrics (held-out set)
------------------------------
  loss        : 0.9164
  accuracy    : 0.8944
  precision   : 0.8941
  recall      : 0.8837
  f1          : 0.8889
  auc         : 0.9474

Model weights saved → /Users/umair/mlops/models/checkpoints/jfcnn.pt
token2idx saved    → /Users/umair/mlops/models/checkpoints/token2idx.json
Embeddings saved   → /Users/umair/mlops/models/checkpoints/embeddings.npz
Train config saved → /Users/umair/mlops/models/checkpoints/train_config.json
